# Notebook 03 — Modelagem

Parte da base limpa e enriquecida produzida no notebook 02 (`venda_espacial.parquet` /
`aluguel_espacial.parquet`) e conduz toda a modelagem: preparação (split, log,
encoding), **baseline definitivo** sobre a base limpa, adição das features espaciais
(distância a estações, e adiante socioeconômico e spatial lag), e comparação com
modelos de ML.

O princípio que rege a preparação continua sendo a prevenção de vazamento: split cedo,
e toda transformação que aprende dos dados ajustada apenas no treino.

In [1]:
import pandas as pd
import numpy as np

# Carrega as bases limpas e enriquecidas (saída do notebook 02)
df_venda   = pd.read_parquet("data/processed/venda_espacial.parquet")
df_aluguel = pd.read_parquet("data/processed/aluguel_espacial.parquet")

print("Venda:  ", df_venda.shape)
print("Aluguel:", df_aluguel.shape)
print("\nColunas:", list(df_venda.columns))

# Confere que a feature de distância veio junto e está sã
print("\nDistância à estação (venda) — resumo:")
print(df_venda["distancia_estacao"].describe()[["min", "50%", "max"]])

# Sanidade: sem NaN nas colunas que vamos usar
print("\nNaN por base — venda:", df_venda.isna().sum().sum(),
      "| aluguel:", df_aluguel.isna().sum().sum())

Venda:   (5936, 17)
Aluguel: (6687, 17)

Colunas: ['Price', 'Condo', 'Size', 'Rooms', 'Toilets', 'Suites', 'Parking', 'Elevator', 'Furnished', 'Swimming Pool', 'New', 'District', 'Negotiation Type', 'Property Type', 'Latitude', 'Longitude', 'distancia_estacao']

Distância à estação (venda) — resumo:
min      18.450136
50%    1153.538874
max    7469.255164
Name: distancia_estacao, dtype: float64

NaN por base — venda: 1276 | aluguel: 595


## 1. Preparação — divisão treino/teste

Sobre a base limpa, refazemos o split treino/teste (80/20, semente fixa), antes de
qualquer transformação que aprenda dos dados. Definimos **explicitamente** as colunas
que entram como features — evitando que colunas descartadas (`Condo`, `Negotiation
Type`, `Property Type`) entrem por descuido. A `distancia_estacao` já vai com a base
desde o split, então não há risco de dessincronia entre features e observações.

In [2]:
from sklearn.model_selection import train_test_split

# Colunas que entram como features (explícito: o que NÃO está aqui, não entra)
# Fora de propósito: Condo (descartado), Negotiation Type (critério), Property Type (constante),
#                    District (vai virar dummies à parte), Latitude/Longitude (insumo espacial, não features cruas)
features_base = [
    "Size", "Rooms", "Toilets", "Suites", "Parking",
    "Elevator", "Furnished", "Swimming Pool", "New",
    "distancia_estacao",   # nova feature espacial
    "District",            # entra agora para ser splitado junto; vira dummies depois
]

def preparar_split(df, nome):
    X = df[features_base].copy()
    y = df["Price"].copy()
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42
    )
    print(f"{nome:8} → treino: {X_train.shape[0]:>5} | teste: {X_test.shape[0]:>5} | colunas: {X_train.shape[1]}")
    return X_train, X_test, y_train, y_test

X_train_v, X_test_v, y_train_v, y_test_v = preparar_split(df_venda, "Venda")
X_train_a, X_test_a, y_train_a, y_test_a = preparar_split(df_aluguel, "Aluguel")

Venda    → treino:  4748 | teste:  1188 | colunas: 11
Aluguel  → treino:  5349 | teste:  1338 | colunas: 11


## 2. Transformações: log e encoding

Reaplicamos sobre a base limpa as transformações já validadas no notebook 02:
- **log** em `Price` (alvo) e `Size`, via `log1p` (relação hedônica multiplicativa,
  coeficiente da área vira elasticidade, normaliza a assimetria à direita);
- **one-hot** do `District`, com `OneHotEncoder` ajustado só no treino
  (`handle_unknown='ignore'` para bairros não vistos).

A `distancia_estacao` permanece como está por ora (sua forma final — crua, log ou
faixas — será decidida ao adicioná-la ao modelo).

In [3]:
from sklearn.preprocessing import OneHotEncoder

def transformar(X_train, X_test, y_train, y_test):
    # --- log do alvo (guarda versão em reais para avaliar em MAPE depois) ---
    y_train_log = np.log1p(y_train)
    y_test_log  = np.log1p(y_test)

    # --- log da área: cria log_Size, remove Size ---
    for X in (X_train, X_test):
        X["log_Size"] = np.log1p(X["Size"])
        X.drop(columns="Size", inplace=True)

    # --- one-hot do District: fit só no treino ---
    enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    enc.fit(X_train[["District"]])
    nomes = enc.get_feature_names_out(["District"])

    def aplicar_encoding(X):
        dummies = pd.DataFrame(enc.transform(X[["District"]]), columns=nomes, index=X.index)
        return pd.concat([X.drop(columns="District"), dummies], axis=1)

    X_train = aplicar_encoding(X_train)
    X_test  = aplicar_encoding(X_test)

    return X_train, X_test, y_train_log, y_test_log

X_train_v, X_test_v, y_train_v_log, y_test_v_log = transformar(X_train_v, X_test_v, y_train_v, y_test_v)
X_train_a, X_test_a, y_train_a_log, y_test_a_log = transformar(X_train_a, X_test_a, y_train_a, y_test_a)

# Confere: matriz numérica, sem NaN, treino e teste com mesmas colunas
print(f"Venda   — treino: {X_train_v.shape} | teste: {X_test_v.shape}")
print(f"Aluguel — treino: {X_train_a.shape} | teste: {X_test_a.shape}")
print("NaN venda:", X_train_v.isna().sum().sum(), "| NaN aluguel:", X_train_a.isna().sum().sum())
print("Colunas iguais (venda)?", list(X_train_v.columns) == list(X_test_v.columns))

Venda   — treino: (4748, 105) | teste: (1188, 105)
Aluguel — treino: (5349, 103) | teste: (1338, 103)
NaN venda: 0 | NaN aluguel: 0
Colunas iguais (venda)? True


## 3. Baseline definitivo e ganho da distância à estação

Dois modelos sobre a base limpa, para isolar o efeito da feature espacial:
1. **Baseline definitivo** — atributos físicos + `District` (sem distância). É o
   marco-zero recalculado na base limpa (sem Jundiaí e sem imóveis com coordenadas corrompidas),
   que substitui o baseline preliminar de 16,5% / 22,4% de MAPE.
2. **Baseline + distância** — o mesmo modelo, porém acrescido de `distancia_estacao`.

A diferença de MAPE entre os dois é o **ganho atribuível à acessibilidade ao
transporte**, medido de forma limpa: mesma base, mesmo split, única diferença é a
presença da feature. Avaliação em reais (MAPE), revertendo a previsão com `expm1`.

In [4]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, r2_score

def avaliar(X_train, X_test, y_train_log, y_test_real, usar_distancia, rotulo):
    # Remove a distância da matriz se for o baseline puro
    if not usar_distancia:
        X_train = X_train.drop(columns="distancia_estacao")
        X_test  = X_test.drop(columns="distancia_estacao")

    modelo = LinearRegression().fit(X_train, y_train_log)
    pred_real = np.expm1(modelo.predict(X_test))  # log → reais

    mape = mean_absolute_percentage_error(y_test_real, pred_real)
    mae  = mean_absolute_error(y_test_real, pred_real)
    r2   = r2_score(y_test_real, pred_real)
    print(f"{rotulo:28} | MAPE: {mape:6.1%} | R²: {r2:.3f} | MAE: R$ {mae:,.0f}")
    return mape

print("=== VENDA ===")
mape_v_base = avaliar(X_train_v, X_test_v, y_train_v_log, y_test_v, False, "Baseline (sem distância)")
mape_v_dist = avaliar(X_train_v, X_test_v, y_train_v_log, y_test_v, True,  "Baseline + distância")

print("\n=== ALUGUEL ===")
mape_a_base = avaliar(X_train_a, X_test_a, y_train_a_log, y_test_a, False, "Baseline (sem distância)")
mape_a_dist = avaliar(X_train_a, X_test_a, y_train_a_log, y_test_a, True,  "Baseline + distância")

print(f"\nGanho da distância — venda:   {(mape_v_base - mape_v_dist)*100:+.2f} p.p.")
print(f"Ganho da distância — aluguel: {(mape_a_base - mape_a_dist)*100:+.2f} p.p.")

=== VENDA ===
Baseline (sem distância)     | MAPE:  17.1% | R²: 0.887 | MAE: R$ 103,675
Baseline + distância         | MAPE:  17.0% | R²: 0.889 | MAE: R$ 103,094

=== ALUGUEL ===
Baseline (sem distância)     | MAPE:  22.3% | R²: 0.735 | MAE: R$ 779
Baseline + distância         | MAPE:  22.2% | R²: 0.733 | MAE: R$ 778

Ganho da distância — venda:   +0.06 p.p.
Ganho da distância — aluguel: +0.06 p.p.


In [ ]:
# Testa a distância em LOG
def avaliar_log_dist(X_train, X_test, y_train_log, y_test_real, rotulo):
    Xtr, Xte = X_train.copy(), X_test.copy()
    Xtr["distancia_estacao"] = np.log1p(Xtr["distancia_estacao"])
    Xte["distancia_estacao"] = np.log1p(Xte["distancia_estacao"])
    modelo = LinearRegression().fit(Xtr, y_train_log)
    pred_real = np.expm1(modelo.predict(Xte))
    mape = mean_absolute_percentage_error(y_test_real, pred_real)
    r2 = r2_score(y_test_real, pred_real)
    print(f"{rotulo:28} | MAPE: {mape:6.1%} | R²: {r2:.3f}")
    return mape

print("=== VENDA ===")
print(f"{'Baseline (sem distância)':28} | MAPE: {mape_v_base:6.1%}")
print(f"{'Baseline + distância crua':28} | MAPE: {mape_v_dist:6.1%}")
mape_v_logdist = avaliar_log_dist(X_train_v, X_test_v, y_train_v_log, y_test_v, "Baseline + log(distância)")

print("\n=== ALUGUEL ===")
print(f"{'Baseline (sem distância)':28} | MAPE: {mape_a_base:6.1%}")
print(f"{'Baseline + distância crua':28} | MAPE: {mape_a_dist:6.1%}")
mape_a_logdist = avaliar_log_dist(X_train_a, X_test_a, y_train_a_log, y_test_a, "Baseline + log(distância)")

=== VENDA ===
Baseline (sem distância)     | MAPE:  17.1%
Baseline + distância crua    | MAPE:  17.0%
Baseline + log(distância)    | MAPE:  17.1% | R²: 0.890

=== ALUGUEL ===
Baseline (sem distância)     | MAPE:  22.3%
Baseline + distância crua    | MAPE:  22.2%
Baseline + log(distância)    | MAPE:  22.3% | R²: 0.731


### Resultado: a distância à estação é redundante com o District

Testada em três formas — crua, log e ausente — a `distancia_estacao` não alterou o MAPE
de forma relevante (venda 17,1% nos três casos; aluguel 22,2–22,3%). O ganho é nulo
(+0,06 p.p.), e o log não recupera sinal, descartando a hipótese de forma funcional
inadequada.

**Achado:** na presença do `District` (one-hot), a distância contínua à estação não
agrega poder preditivo. A localização opera majoritariamente na escala do bairro — as
dummies de District já absorvem a acessibilidade ao transporte —, e a variação
intra-bairro de distância é pequena demais para mover o modelo. Não é falha de
construção (a feature foi auditada e está correta); é redundância genuína com a
localização categórica.

**Implicação:** o ganho espacial, se houver, virá do **spatial lag** (Bloco 3), que
captura o nível de preço do entorno imediato numa granularidade mais fina que o bairro
— escala que nem o District nem a distância cobrem.

### Decisão sobre a feature de distância

**No baseline linear:** a `distancia_estacao` é **excluída**. Testes mostraram ganho
nulo (+0,06 p.p. em qualquer forma), e, sendo redundante com o `District` (ambos
capturam localização), incluí-la introduziria **multicolinearidade** desnecessária —
inflando a variância dos coeficientes sem retorno preditivo, contra o princípio de
parcimônia da tradição NBR. O modelo linear ideal usa atributos físicos + `District`.

**Nos modelos de árvore (fase de ML):** a feature será **reintroduzida e testada** (com
vs. sem), pelo mesmo protocolo de robustez. Justificativa: árvores não sofrem de
multicolinearidade (escolhem a melhor variável por nó) e captam não-linearidades e
interações (ex.: distância relevante apenas em bairros periféricos) que a regressão
linear não expressa. A feature permanece na base (`*_espacial.parquet`), guardada para
essa reavaliação.

## 4. Enriquecimento socioeconômico

Segunda feature espacial: a **renda média domiciliar** do entorno de cada imóvel, do
Censo 2010 (respeitando a contemporaneidade temporal do projeto), na granularidade de
**área de ponderação** — a unidade fina onde a renda do Censo é oficialmente divulgada
(mais detalhada que o distrito, mas agregada o suficiente para o IBGE liberar renda sem
ferir sigilo estatístico).

**Fonte:** dataset *Project Atlas - São Paulo* [(Kaggle, por Mateus Picanço)](https://www.kaggle.com/datasets/mateuspicanco/sao-paulo-geospatial-features), que agrega
dados oficiais do GeoSampa e do IBGE Censo 2010 em 310 áreas de ponderação da capital,
com geometria (polígonos). Procedência documentada e auditável.

**Por que renda (e não IDH ou distância):** a renda é a variável socioeconômica mais
ligada a preço imobiliário na literatura hedônica, e — diferente da distância à estação
— tem forte poder discriminante entre bairros. Sendo de escala fina (intra-distrito),
ela *pode* agregar onde a distância (escala do bairro) falhou. Hipótese a medir.

**Procedimento:** join espacial *point-in-polygon* — cada imóvel recebe a renda da área
de ponderação que o contém.

### 4.1 Carregamento e validação

Carregamos o arquivo de áreas de ponderação com `geopandas` (que decodifica a geometria
corretamente) e fazemos duas verificações antes de usar os dados: inspecionamos o
sistema de coordenadas (CRS) dos polígonos — necessário para o join espacial adiante — e
validamos a variável de renda, auditando sua distribuição e procedência. Não se usa
feature de terceiro sem confirmar o que ela é.

In [15]:
import geopandas as gpd

# Carrega com geopandas, que decodifica a geometria corretamente
gdf_ap = gpd.read_parquet("data/raw/tb_area_of_ponderation.parquet")

print("Áreas de ponderação:", gdf_ap.shape[0])
print("CRS:", gdf_ap.crs)
print("Tipo de geometria:", gdf_ap.geometry.geom_type.unique())

Áreas de ponderação: 310
CRS: None
Tipo de geometria: <ArrowStringArray>
['Polygon', 'MultiPolygon']
Length: 2, dtype: str


In [17]:
# Inspeciona as coordenadas de um polígono para inferir o CRS real
poligono = gdf_ap.geometry.iloc[0]
print("Bounds do primeiro polígono (minx, miny, maxx, maxy):")
print(poligono.bounds)

print("\nBounds de toda a base:")
print(gdf_ap.total_bounds)

Bounds do primeiro polígono (minx, miny, maxx, maxy):
(-46.731457834553574, -23.620916222146832, -46.71949277744977, -23.61147419784277)

Bounds de toda a base:
[-46.82628559 -24.00841351 -46.36517338 -23.35627846]


O dataset traz 310 áreas de ponderação com geometria de polígonos, mas sem o CRS
declarado (`None`). Pela magnitude das coordenadas (longitude ~−46, latitude ~−23/−24)
e pelos bounds coincidentes com o município de São Paulo, identificamos que os polígonos
estão em **WGS 84 (EPSG:4326)** — lat/long, o mesmo sistema dos imóveis. Declaramos o
CRS explicitamente (sem reprojetar, pois já coincide). A renda média domiciliar foi
validada (R$ 987 a 14.507, razão ~15x), confiável e com procedência no Censo 2010.

In [16]:
# Validação da variável de renda escolhida: renda média domiciliar
RENDA = "ponderation_area_average_household_income"

print("Distribuição da renda média domiciliar (R$):")
print(gdf_ap[RENDA].describe().round(0))

# Teste de poder discriminante: razão entre a área mais rica e a mais pobre
renda_max = gdf_ap[RENDA].max()
renda_min = gdf_ap[RENDA].min()
print(f"\nMais rica:  R$ {renda_max:,.0f}")
print(f"Mais pobre: R$ {renda_min:,.0f}")
print(f"Razão rico/pobre: {renda_max/renda_min:.1f}x")

Distribuição da renda média domiciliar (R$):
count      310.0
mean      3449.0
std       2569.0
min        987.0
25%       1728.0
50%       2356.0
75%       4208.0
max      14507.0
Name: ponderation_area_average_household_income, dtype: float64

Mais rica:  R$ 14,507
Mais pobre: R$ 987
Razão rico/pobre: 14.7x


A renda média domiciliar varia de R$ 987 (área mais pobre) a R$ 14.507 (mais rica),
uma razão de ~15x — forte poder discriminante, coerente com a desigualdade de São Paulo
e em contraste com a distância à estação, que mal variava. A mediana fica em ~R$ 2.356.
A variável é confiável e bem definida; escolhida sobre per capita por ser a mais ligada
ao poder de compra do domicílio (quem adquire o imóvel é o lar, não o indivíduo).
Procedência: IBGE Censo 2010, via Project Atlas.

### 4.2 Join espacial: renda por imóvel

Cada imóvel recebe a renda da área de ponderação que o contém — um join espacial
*point-in-polygon*. Transformamos os imóveis em pontos geográficos, declaramos o CRS das
áreas de ponderação (EPSG:4326, já alinhado), e usamos `sjoin` com predicado `within`
(o imóvel está *dentro* do polígono). A renda resultante vira a feature
`renda_area`, calculada igualmente para venda e aluguel (feature geométrica, sem uso do
alvo — sem risco de leakage).

In [18]:
# 1. Declara o CRS correto nas áreas de ponderação
gdf_ap = gdf_ap.set_crs("EPSG:4326")

def adicionar_renda(df, nome):
    # 2. Transforma os imóveis em pontos geográficos (lat/long)
    gdf_imoveis = gpd.GeoDataFrame(
        df.copy(),
        geometry=gpd.points_from_xy(df["Longitude"], df["Latitude"]),
        crs="EPSG:4326"
    )

    # 3. Join espacial: cada imóvel recebe os dados da área de ponderação que o contém
    juntado = gpd.sjoin(
        gdf_imoveis,
        gdf_ap[[RENDA, "geometry"]],
        how="left",
        predicate="within"
    )

    # 4. Remove duplicatas (imóvel em cima de divisa pode casar com 2 áreas)
    juntado = juntado[~juntado.index.duplicated(keep="first")]

    # 5. Renomeia a coluna de renda e devolve só ela, alinhada ao índice original
    return juntado[RENDA].rename("renda_area")

df_venda["renda_area"]   = adicionar_renda(df_venda, "Venda")
df_aluguel["renda_area"] = adicionar_renda(df_aluguel, "Aluguel")

# Conferência: quantos imóveis ficaram SEM renda (caíram fora de qualquer polígono)?
print("Imóveis sem renda (venda):  ", df_venda["renda_area"].isna().sum(), "de", len(df_venda))
print("Imóveis sem renda (aluguel):", df_aluguel["renda_area"].isna().sum(), "de", len(df_aluguel))
print("\nRenda atribuída — venda (resumo):")
print(df_venda["renda_area"].describe().round(0))

Imóveis sem renda (venda):   12 de 5936
Imóveis sem renda (aluguel): 17 de 6687

Renda atribuída — venda (resumo):
count     5924.0
mean      4603.0
std       2750.0
min        998.0
25%       2465.0
50%       3588.0
75%       5903.0
max      13288.0
Name: renda_area, dtype: float64


### 4.3 Tratamento dos imóveis órfãos (vizinho mais próximo)

29 imóveis (12 venda, 17 aluguel) caíram fora de qualquer área de ponderação — em
divisas ou pequenos vãos da malha. Como têm coordenada válida (não são corrompidos),
preenchemos sua renda com a da **área de ponderação mais próxima**, via `sjoin_nearest`.
É uma estimativa localizada e honesta: um imóvel na borda de uma área tem renda de
entorno semelhante à dela. O cálculo do "mais próximo" é feito em CRS métrico (UTM
31983), pois distância em graus distorce.

In [19]:
def preencher_renda_orfaos(df, nome):
    # Identifica os órfãos (renda NaN após o join within)
    orfaos = df[df["renda_area"].isna()].copy()
    if len(orfaos) == 0:
        print(f"{nome}: sem órfãos.")
        return df["renda_area"]

    # Pontos dos órfãos e polígonos, ambos em UTM (métrico) para medir distância correta
    gdf_orfaos = gpd.GeoDataFrame(
        orfaos,
        geometry=gpd.points_from_xy(orfaos["Longitude"], orfaos["Latitude"]),
        crs="EPSG:4326"
    ).to_crs(epsg=31983)
    ap_utm = gdf_ap[[RENDA, "geometry"]].to_crs(epsg=31983)

    # Join pela área de ponderação MAIS PRÓXIMA (não a que contém, que não existe p/ órfãos)
    preenchidos = gpd.sjoin_nearest(gdf_orfaos, ap_utm, how="left")
    preenchidos = preenchidos[~preenchidos.index.duplicated(keep="first")]

    # Atualiza a coluna renda_area só nos índices órfãos
    renda = df["renda_area"].copy()
    renda.loc[orfaos.index] = preenchidos[RENDA]
    print(f"{nome}: {len(orfaos)} órfãos preenchidos pela área mais próxima.")
    return renda

df_venda["renda_area"]   = preencher_renda_orfaos(df_venda, "Venda")
df_aluguel["renda_area"] = preencher_renda_orfaos(df_aluguel, "Aluguel")

# Conferência: não deve sobrar nenhum NaN
print("\nNaN restantes — venda:", df_venda["renda_area"].isna().sum(),
      "| aluguel:", df_aluguel["renda_area"].isna().sum())

Venda: 12 órfãos preenchidos pela área mais próxima.
Aluguel: 17 órfãos preenchidos pela área mais próxima.

NaN restantes — venda: 0 | aluguel: 0


### 4.4 Teste: o ganho da renda sobre o baseline

Com a renda completa, medimos seu efeito pelo mesmo protocolo da distância: baseline
(atributos + District) vs. baseline + renda, mesma base e split, isolando o efeito da
feature. Testamos a renda em duas formas — crua e em log (a renda tem cauda à direita,
então o log pode capturar melhor o efeito decrescente do poder aquisitivo sobre o preço).

In [21]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_percentage_error, r2_score

def testar_renda(X_train, X_test, y_train_log, y_test_real, df_origem, forma, rotulo):
    Xtr, Xte = X_train.copy(), X_test.copy()
    # Puxa a renda pelo índice (alinha cada imóvel à sua renda)
    Xtr["renda_area"] = df_origem.loc[Xtr.index, "renda_area"]
    Xte["renda_area"] = df_origem.loc[Xte.index, "renda_area"]
    if forma == "log":
        Xtr["renda_area"] = np.log1p(Xtr["renda_area"])
        Xte["renda_area"] = np.log1p(Xte["renda_area"])

    modelo = LinearRegression().fit(Xtr, y_train_log)
    pred = np.expm1(modelo.predict(Xte))
    mape = mean_absolute_percentage_error(y_test_real, pred)
    r2 = r2_score(y_test_real, pred)
    print(f"{rotulo:32} | MAPE: {mape:6.1%} | R²: {r2:.3f}")
    return mape

print("=== VENDA ===")
print(f"{'Baseline (sem renda)':32} | MAPE: {mape_v_base:6.1%}")
testar_renda(X_train_v, X_test_v, y_train_v_log, y_test_v, df_venda, "crua", "Baseline + renda (crua)")
testar_renda(X_train_v, X_test_v, y_train_v_log, y_test_v, df_venda, "log",  "Baseline + renda (log)")

print("\n=== ALUGUEL ===")
print(f"{'Baseline (sem renda)':32} | MAPE: {mape_a_base:6.1%}")
testar_renda(X_train_a, X_test_a, y_train_a_log, y_test_a, df_aluguel, "crua", "Baseline + renda (crua)")
testar_renda(X_train_a, X_test_a, y_train_a_log, y_test_a, df_aluguel, "log",  "Baseline + renda (log)")

=== VENDA ===
Baseline (sem renda)             | MAPE:  17.1%
Baseline + renda (crua)          | MAPE:  16.8% | R²: 0.892
Baseline + renda (log)           | MAPE:  16.7% | R²: 0.891

=== ALUGUEL ===
Baseline (sem renda)             | MAPE:  22.3%
Baseline + renda (crua)          | MAPE:  22.1% | R²: 0.734
Baseline + renda (log)           | MAPE:  22.2% | R²: 0.733


0.22152038655393327